# Episode 4 — Control Flow with JIT

**Student workbook** · code along with the video.

Python `if` / `for` / `while` work eagerly, but **`jit` traces one path** through your control-flow graph. When the path depends on **input values**, you need `static_argnames`, `lax.cond`, or `lax.*_loop`.

| | |
|---|---|
| **Chapter** | 1.4 · Part I — Pure JAX |
| **Prereq** | Episodes 1–3 |
| **Next** | Episode 5 — pytrees and SGD |

**Source:** [Control flow and logical operators with JIT](https://docs.jax.dev/en/latest/control-flow.html)

In [ ]:
# your code here


## Control flow under `jit`

Eagerly, JAX behaves like NumPy. Under **`jit`**, Python control flow is evaluated at **compile time** — the compiled function is **one path** through the graph. Logical operators short-circuit the same way.

If the path depends on **input values**, tracing fails by default. It may depend on **shape/dtype** — then JAX **recompiles** on new shapes.

### What works

In [ ]:
# your code here


In [ ]:
# your code here


### What fails (value-dependent branch)

In [ ]:
# your code here


In [ ]:
# your code here


### Why?

`jit` traces on **`ShapedArray`** — shape + dtype, not a specific value. For `if x < 3`, the predicate is `{True, False}` abstractly; Python cannot pick a branch, so tracing stops.

**Tradeoff:** more abstract traces → fewer recompiles, but stricter rules on Python control flow.

### Branches on `.shape` vs `static_argnames`

Under `jit`, JAX traces on **`ShapedArray`** — shape and dtype are concrete. Branch on `.shape` and JAX compiles **one path per shape** (recompiles when shape changes).

For branches on a **Python scalar's value** (not shape), use `static_argnames` (below).

In [ ]:
# your code here


In [ ]:
# your code here


With `static_argnames='n'`, the loop is **statically unrolled** at trace time (fine for small `n`; disastrous if `n` changes every call).

⚠️ **`static_argnames` can be handy when `length` rarely changes, but costly if it changes a lot.**

### Value-dependent shapes

Same issue when **array shapes** depend on argument **values** (shape-specialization on dtype/shape alone is OK):

In [ ]:
# your code here


In [ ]:
# your code here


In [ ]:
# your code here


### Side effects inside `jit`

`print` runs at trace time and shows **tracers**, not concrete values. For debug output inside compiled code, use [`jax.debug.print`](https://docs.jax.dev/en/latest/_autosummary/jax.debug.print.html) (Episode 2).

In [ ]:
# your code here


## Structured control flow primitives

When you want **traceable** control flow **without** recompiling on every branch — and **without** unrolling large loops — use `jax.lax`:

| Primitive | Role |
|---|---|
| `lax.cond` | differentiable `if` on a scalar predicate |
| `lax.while_loop` | `while` with runtime stop condition |
| `lax.fori_loop` | counted `for`; XLA may lower to `scan` or `while_loop` |
| `lax.scan` | fold/map/scan with per-step inputs |

### `lax.cond`

In [ ]:
# your code here


Related: [`lax.select`](https://docs.jax.dev/en/latest/_autosummary/jax.lax.select.html) (batched choices as arrays), [`lax.switch`](https://docs.jax.dev/en/latest/_autosummary/jax.lax.switch.html) (multi-branch). NumPy-style: `jnp.where`, `jnp.piecewise`, `jnp.select`.

### `lax.while_loop`

In [ ]:
# your code here


### `lax.fori_loop`

In [ ]:
# your code here


### Summary — `jit` vs `grad`

| construct | `jit` | `grad` |
|---|---|---|
| `if` | ❌ (value-dependent) | ✔ |
| `for` | ✔* | ✔ |
| `while` | ✔* | ✔ |
| `lax.cond` | ✔ | ✔ |
| `lax.while_loop` | ✔ | fwd only |
| `lax.fori_loop` | ✔ | fwd only† |
| `lax.scan` | ✔ | ✔ |

\* loop bound must be **value-independent** (compile-time / shape-based) — otherwise unrolls or fails.

† `fori_loop` is rev-mode differentiable when endpoints are static.

## Logical operators

Use **`jnp.logical_and` / `logical_or` / `logical_not`** (or bitwise `&` `|` `~`) under `jit`. Unlike Python `and`/`or`, they **do not short-circuit** — both sides are evaluated.

In [ ]:
# your code here


In [ ]:
# your code here


In [ ]:
# your code here


## Python control flow + `grad` (no `jit`)

The constraints above apply to **`jit` only**. **`grad`** on pure Python functions with `if`/`for` works fine — Autograd-style:

In [ ]:
# your code here


> **Key insight:** Under `jit`, value-dependent Python control flow fails or forces recompilation via `static_argnames`. For dynamic branches and loops inside one compiled graph, use **`lax.cond`**, **`lax.while_loop`**, **`lax.fori_loop`**, or **`lax.scan`**.

---
## Exercise

1. Write a `@jit` function with `if x < 0` that fails; fix it with `static_argnames` or `lax.cond`.
2. Write `example_fun(length, val)` with `jit` and `static_argnames='length'`; call it with two different lengths.
3. Implement a counted sum with `lax.fori_loop` and a runtime stop with `lax.while_loop`.
4. Replace `(x > 0) and (x < 3)` with `jnp.logical_and` under `jit`.



In [ ]:
# your code here


**Next:** Episode 5 — pytrees and SGD.